# Demo 4jun25

## Programmatic methods to sample & compare NS and SRNS points

In [1]:
# Custom modules with their classes inside
import behaviors  # noqa: I001
import no_signaling_sets

import numpy as np

import sys
sys.path.append('../code_legacy')
import  extract_hyperplanes # Unfinished module# type: ignore # noqa: I001


## Experiment parameters

In [2]:
delta = 2 # Number of outputs a,b
m = 2     # Number of inputs x,y

## Get a non-SRNS point from database

In [3]:
data = np.load("../data/non_srns/non_srns_points.npy")
example_point = data[np.random.randint(len(data))]

print("Example point as vector:", example_point)
print("\n\n")
print("Example point as behavior:")
behavior = behaviors.RoutedBehavior(delta, m, example_point)
print(behavior)

Example point as vector: [0.15286655 0.36043441 0.4260697  0.35935521 0.26221742 0.05464956
 0.14640626 0.21312075 0.50691609 0.12455818 0.23371294 0.12563738
 0.07799994 0.46035785 0.1938111  0.30188666 0.25792646 0.07058864
 0.14760937 0.0296975  0.15715752 0.34449534 0.42486659 0.54277846
 0.08553607 0.3667307  0.19585316 0.40762184 0.49937996 0.21818533
 0.23167088 0.0199022 ]



Example point as behavior:
Behavior:
Short path (z=S):
[[0.15286655 0.36043441 0.4260697  0.35935521]
 [0.26221742 0.05464956 0.14640626 0.21312075]
 [0.50691609 0.12455818 0.23371294 0.12563738]
 [0.07799994 0.46035785 0.1938111  0.30188666]]
Long path (z=L) :
[[0.25792646 0.07058864 0.14760937 0.0296975 ]
 [0.15715752 0.34449534 0.42486659 0.54277846]
 [0.08553607 0.3667307  0.19585316 0.40762184]
 [0.49937996 0.21818533 0.23167088 0.0199022 ]]
------------


### Elementary tests on behaviors

In [4]:
print(f"Coordinates are positive                      : {behavior.positivity()}")
print(f"Coordinates are normalized                    : {behavior.normalization()}")
print(f"Coordinates verify the no-signaling conditions: {behavior.no_signaling()}")
print()
print("Aggregated tests (checks all previous tests):")
print(f"  Normalization                               : {behavior.is_normalized()}")
print(f"  No-signaling                                : {behavior.is_no_signaling()}")

Coordinates are positive                      : True
Coordinates are normalized                    : True
Coordinates verify the no-signaling conditions: True

Aggregated tests (checks all previous tests):
  Normalization                               : True
  No-signaling                                : True


## Instanciate a set to test for belonging in SRNS

In [5]:
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)
srns_set

### Pipelined belonging test

In [6]:
print(f"Does SRNS set contains the example point: {srns_set.is_in_set(behavior)}")

Does SRNS set contains the example point: False


### Main steps to belonging test

#### [1] Get the equation to test for belonging in matrix form

In [7]:
A,b = srns_set.get_equations(behavior)

with np.printoptions(threshold=np.inf, linewidth=np.inf, precision=2): # type: ignore
    print("A matrix:")
    print(A[:-4])
    print("Last 4 rows of A matrix")
    print(A[-4:])
    print()
    print("b vector:")
    print(b)

A matrix:
[[ 0.1   1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.11  0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.18  0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.11  0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.01  0.    0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0. 

#### [2] Solve the linear program

In [8]:
from scipy.optimize import OptimizeResult

result: OptimizeResult = srns_set.lp_test(behavior)

In [9]:
print(f"Alpha value for  alpha*p + (1-alpha)*I  : {-result.fun}", "< 1" if -result.fun < 1 else ">= 1")  # noqa: E501

print()

latent_q_vector = result.x[1:]
latent_behavior = behaviors.LatentSRNSBehavior(delta, m, latent_q_vector)
print("Latent behavior:")
print(latent_behavior)
print("Latent behavior is no-signaling: ", latent_behavior.no_signaling())

Alpha value for  alpha*p + (1-alpha)*I  : 0.9618598531563328 < 1

Latent behavior:
Behavior:
Short path (z=S):
[[0.15657123 0.35622243 0.41935437 0.35518438]
 [0.26175145 0.06210026 0.15035734 0.21452733]
 [0.49711727 0.12934255 0.23433413 0.13038059]
 [0.08456004 0.45233477 0.19595415 0.29990769]]
Long path (z=L) :
[[0.07743141 0.        ]
 [0.18019273 0.15151456]
 [0.         0.03809987]
 [0.16069854 0.38009729]
 [0.09180875 0.16924016]
 [0.         0.02867817]
 [0.27046982 0.23236996]
 [0.21939874 0.        ]]
------------
Latent behavior is no-signaling:  True


In [10]:
def format_lambda_to_hyperplane(lam: np.ndarray) -> str:
    return str(lam[:16]) + str(lam[16:32]) + str(lam[32:]) 

In [11]:
_, _, lambda_var = srns_set.is_facet_hyperplane(behavior)
hyperplanes_extractor = extract_hyperplanes.HyperplanesExtractor(delta, m, list(data))
print("Corresponding lambda variable:")
with np.printoptions(threshold=np.inf, precision=2):  # type: ignore
    print(lambda_var)
print(len(lambda_var), "is the number of coordinates in the dual variable")
print()

hyperplane = hyperplanes_extractor.scale_down_vector(lambda_var)
print("Rescaled and sliced to the hyperplane size:")
print(format_lambda_to_hyperplane(hyperplane))

Corresponding lambda variable:
[ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    1.92  0.    0.    0.    1.92 -1.92
  0.    0.    0.    0.   -1.92  1.92  1.92  0.    0.    0.    1.92  0.  ]
36 is the number of coordinates in the dual variable

Rescaled and sliced to the hyperplane size:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0][ 0  0  1  0  0  0  1 -1  0  0  0  0 -1  1  1  0][0 0 1 0]


In [12]:
np.dot(lambda_var[:32], behavior.get_vector())


np.float64(-0.03814014684366729)

In [13]:
np.dot(lambda_var[:32], behaviors.completely_mixed_behavior.get_vector())

np.float64(0.9618598531563327)

# Sampling

In [14]:
import samplers

In [15]:
sampler = samplers.NoSignalingSampler(delta, m, True)

In [16]:
sampled_vec = sampler.sample()
print("Sampled vector:")
print(sampled_vec)

2025-06-06 10:41:13.104 | SUCCESS  | samplers:sample_multiple:116 - Samples shape: (1, 32)


Sampled vector:
Behavior:
Short path (z=S):
[[0.22244426 0.50850048 0.32359441 0.34497699]
 [0.36936153 0.08330531 0.25060457 0.229222  ]
 [0.21266511 0.09370785 0.11151496 0.25723134]
 [0.19552909 0.31448635 0.31428605 0.16856967]]
Long path (z=L) :
[[0.12606199 0.45197474 0.27184978 0.23294336]
 [0.46574381 0.13983105 0.30234921 0.34125563]
 [0.34782821 0.09203345 0.20204042 0.31106483]
 [0.060366   0.31616076 0.22376059 0.11473618]]
------------


In [17]:
srns_set

In [18]:
identity = behaviors.completely_mixed_behavior


In [19]:
result = srns_set.lp_test(identity)

In [20]:
-result.fun

2.0

# Post-demo : testing on quantum NS distributions

In [ ]:
from qutip import Qobj, basis, expect, ket2dm, qeye, sigmax, sigmaz, tensor

# AI GENRATED CODE

def projectors(op: Qobj):
    """Return projectors Π_{+1} and Π_{-1} for a Hermitian observable with eigenvalues ±1"""
    eigvals, eigvecs = op.eigenstates()
    proj_dict = {}
    for val, vec in zip(eigvals, eigvecs):
        key = int(np.sign(val)) if val != 0 else +1  # Disambiguate 0 as +1
        proj_dict[key] = ket2dm(vec)
    return proj_dict[+1], proj_dict[-1]

def bell_conditional_distribution(
    alice_ops: list[Qobj],
    bob_ops: list[Qobj]
) -> dict[tuple[int, int], dict[tuple[int, int], float]]:
    """Compute P(a, b | x, y) for all a,b ∈ {±1}, x in Alice ops, y in Bob ops"""

    # Create Φ⁺ state
    ket0 = basis(2, 0)
    ket1 = basis(2, 1)
    phi_plus = (tensor(ket0, ket0) + tensor(ket1, ket1)).unit()
    rho = ket2dm(phi_plus)

    outcomes = [-1, +1]
    distribution = dict()

    for x_index, Ax in enumerate(alice_ops):
        for y_index, By in enumerate(bob_ops):
            # Projectors Π^A_a ⊗ Π^B_b
            Pa_pos, Pa_neg = projectors(Ax)
            Pb_pos, Pb_neg = projectors(By)

            Pxy = dict()
            for a in outcomes:
                Pa = Pa_pos if a == +1 else Pa_neg
                for b in outcomes:
                    Pb = Pb_pos if b == +1 else Pb_neg
                    M = tensor(Pa, Pb)
                    prob = (M * rho).tr().real
                    Pxy[(a, b)] = round(prob, 10)  # Clean rounding
            distribution[(x_index, y_index)] = Pxy

    return distribution
